# Somo 02 - Kuchunguza Mfumo wa Wakala wa Microsoft

**Mfumo wa Wakala wa Microsoft (MAF)** ni mfumo umeingiliana wa kujenga mawakala wa AI. Inatoa usanifu safi, unaoweza kuundwa na vipengele vinne vya msingi:

- **Mteja** – huunganishwa na sehemu ya mwisho ya modeli ya AI na kushughulikia mawasiliano
- **Wakala** – huambatisha mteja na maagizo na ufafanuzi wa zana
- **Zana** – huongeza uwezo wa wakala kwa kazi maalum ambazo modeli inaweza kuita
- **Kikao** – huhifadhi historia ya mazungumzo kwa mwingiliano wa mizunguko mingi

Katika somo hili, tutajenga **wakala wa uhifadhi wa safari** ambaye anathibitisha upatikanaji wa marudio kwa kutumia dhana hizi.


## Usanidi


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Kuelewa Msingi wa Miundombinu ya Wakala

Msingi wa Microsoft Agent Framework unafuata miundo ya tabaka:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Mteja** – `FoundryChatClient` huunganishwa na usambazaji wa Azure OpenAI. Hushughulikia uthibitishaji, uundaji wa maombi, na uchambuzi wa majibu.
2. **Wakala** – Hutengenezwa kutoka kwa mteja kupitia `provider.create_agent()`, wakala huunganisha upatikanaji wa mfano pamoja na maelekezo (onyo la mfumo) na zana.
3. **Zana** – Kazi za Python zilizo na alama ya `@tool` ambazo wakala anaweza kuitumia kufanya vitendo au kupata data.
4. **Kikao** – Kitu cha `AgentSession` (kilichotengenezwa kupitia `agent.create_session()`) kinachohifadhi historia ya mazungumzo, kuwezesha mazungumzo ya mizunguko mingi ambapo wakala anakumbuka muktadha wa awali.

Hebu tujenge kila tabaka hatua kwa hatua.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Kuongeza Zana kwa kutumia Mpaniaji @tool

Zana huruhusu mawakala kuchukua hatua zaidi ya kuzalisha maandishi. Mpaniaji `@tool` hubadilisha kipengele cha kawaida cha Python kuwa kitu ambacho wakala anaweza kuitisha.

Mambo muhimu:
- Tumia `Annotated[type, "description"]` ili mfano uelewe kila parameta.
- Msururu wa maelezo (docstring) huwa maelezo ya zana ambayo mfano unaona.
- `approval_mode="never_require"` ina maana zana inatumika moja kwa moja bila uthibitishaji kutoka kwa mtumiaji.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Kuunda Wakala akiwa na Zana

Sasa tunachanganya mteja, maagizo, na zana kuwa wakala. `maagizo` hutumika kama mwito wa mfumo — huainisha tabia na utu wa wakala.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Mazungumzo ya Mzunguko-Mwingi kwa Kufanya Vikao

`AgentSession` (inayoundwa kupitia `agent.create_session()`) inafuatilia ujumbe wote katika mazungumzo. Kwa kupitisha kikao kile kile kwa kila mwito wa `agent.run()`, wakala anaweza kupata historia nzima ya mazungumzo na kurejelea ujumbe wa awali.

Tunapita `tools=[check_destination_availability]` ili wakala aweze kuita kiongozi wetu wa upatikanaji katika kila mzunguko.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Muhtasari

Katika somo hili ulichunguza nguzo nne za Mfumo wa Wakala wa Microsoft:

| Dhana | Uliyojifunza |
|---------|------------------|
| **Mteja** | `FoundryChatClient` huunganishwa na Azure OpenAI kwa uthibitisho wa kutumia cheti |
| **Wakala** | `provider.create_agent()` huunganisha muunganisho wa mfano na maelekezo na jina |
| **Zana** | Dekorera `@tool` huonyesha shughuli za Python ili wakala aitwe |
| **Kikao** | `agent.create_session()` hudumisha historia ya mazungumzo kupitia mizunguko mingi |

Vipengele hivi vya msingi vinajengwa pamoja kuunda mawakala wanaoweza kuendesha mazungumzo ya kawaida, kuita shughuli za nje, na kudumisha muktadha — msingi wa mifumo ya wakala yenye kina zaidi katika masomo yajayo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
